In [1]:
import os
import numpy as np
import warnings 
import pandas as pd 

from datetime import datetime
import xarray as xr
import xskillscore as xs

from xcdat.dataset import open_dataset
from xcdat.bounds import create_bounds
from xcdat.dataset import open_mfdataset

from ObservationDataUtil import ( 
     ObsDataReader,
    
)

from ModelDataUtil import (
    ModelDataReader,
)

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [2]:
class EnsembleMetricEvaluator:
    def __init__(self, 
                 obs, 
                 model_dict, 
                 component, 
                 ref_dataset, 
                 var, 
                 vunit, 
                 output_path):
        self.obs_data = obs
        self.model_dict = model_dict
        self.component = component
        self.reference = ref_dataset
        self.var_name = var
        self.var_unit = vunit
        self.output_path = output_path

        self.results = self.derive_metrics_data()

    def _generate_coslat_weight(self, da: xr.DataArray) -> xr.DataArray:
        """
        Generate cosine-latitude weights broadcasted to the shape of the input DataArray.

        Parameters
        ----------
        da : xr.DataArray
            Input data array with 'lat' dimension or coordinate.

        Returns
        -------
        xr.DataArray
            Cosine-latitude weights broadcast to the shape of da.
        """
        if 'lat' not in da.coords:
            raise ValueError("Latitude coordinate 'lat' not found in DataArray.")

        weights = np.cos(np.deg2rad(da['lat']))
    
        # If 'lat' is not a dimension but just a coordinate, make it a dimension
        if 'lat' not in da.dims:
            weights = weights.expand_dims(dim='lat')
    
        # Broadcast to match da shape
        weights, _ = xr.broadcast(weights, da)
    
        return weights

    def _get_ensemble_dim(self, da: xr.DataArray) -> str:
        for dim in da.dims:
            if dim in ['ensemble', 'member']:
                return dim
        raise ValueError("No ensemble dimension found (expected 'ensemble' or 'member').")
        
    def bootstrap_ci_map(self, model, obs, metric_fn, ens_dim = "ensemble", n_boot=1000, alpha=0.05):
        """
        Compute metric with bootstrap significance test.

        Parameters:
            model (xr.DataArray): [ensemble, lat, lon]
            obs (xr.DataArray): [lat, lon]
            metric_fn (callable): Function(model_subset, obs) -> xr.DataArray[lat, lon]
            n_boot (int): Number of bootstrap resamples
            alpha (float): Significance level

        Returns:
            metric (xr.DataArray): Mean metric
            mask (xr.DataArray): Boolean mask where metric is significant
            ci (tuple): Lower and upper confidence interval (xr.DataArray)
        """
        bootstrap_metrics = []

        for i in range(n_boot):
            # Resample with replacement along the ensemble dimension
            resample = model.sel({ens_dim: np.random.choice(model[ens_dim].values, size=model.sizes[ens_dim], replace=True)})
            metric = metric_fn(resample, obs, ens_dim = ens_dim)
            bootstrap_metrics.append(metric)
            
        boot_maps = np.stack([m.values for m in bootstrap_metrics], axis=0)
        lower = np.nanpercentile(boot_maps, 100 * alpha / 2, axis=0)
        upper = np.nanpercentile(boot_maps, 100 * (1 - alpha / 2), axis=0)
        mean_metric = metric_fn(model, obs, ens_dim = ens_dim)
        mask = ((mean_metric < lower) | (mean_metric > upper)).astype(int)
        
        lower_da = xr.DataArray(lower, dims=['lat', 'lon'], coords={'lat': obs.lat, 'lon': obs.lon})
        upper_da = xr.DataArray(upper, dims=['lat', 'lon'], coords={'lat': obs.lat, 'lon': obs.lon})
        signif_mask = xr.DataArray(mask, dims=['lat', 'lon'], coords={'lat': obs.lat, 'lon': obs.lon})
        
        return lower_da, upper_da, signif_mask

    def bias_map_fn(self, model, obs, ens_dim="member"):
        # Drop ensemble dim from obs if accidentally present (e.g., from broadcasting)
        if ens_dim in obs.dims:
            obs = obs.mean(dim=ens_dim, skipna=True)  # Or .isel({ens_dim: 0}) if you expect one value
        model = model.transpose(ens_dim,'lat','lon')
        obs = obs.transpose('lat','lon')
        return model.mean(dim=ens_dim, skipna=True) - obs

    def rmse_map_fn(self, model, obs, ens_dim="member"):
        model = model.transpose(ens_dim, "lat", "lon")
        # Drop ensemble dim from obs if accidentally present (e.g., from broadcasting)
        if ens_dim in obs.dims:
            obs = obs.mean(dim=ens_dim, skipna=True)  # Or .isel({ens_dim: 0}) if you expect one value
        # Broadcast obs to model's shape
        obs = obs.expand_dims({ens_dim: model[ens_dim]})
        obs = obs.transpose(ens_dim, "lat", "lon")
        return xs.rmse(model, obs, dim=ens_dim, skipna=True)

    def spread_map_fn(self, model, obs=None, ens_dim='ensemble'):
        model = model.transpose(ens_dim,'lat','lon')
        return model.std(dim=ens_dim, skipna=True)

    def crps_map_fn(self, model, obs, ens_dim='ensemble'):
        # If obs accidentally has an ensemble dimension, remove it
        if ens_dim in obs.dims:
            obs = obs.mean(dim=ens_dim, skipna=True)

        # Ensure both are lat-lon aligned
        model = model.transpose(ens_dim, "lat", "lon")
        obs = obs.transpose("lat", "lon")
        # Compute CRPS (no dim specified; shape is [lat, lon])
        crps = xs.crps_ensemble(obs, model, member_dim=ens_dim, dim=[])
        return crps.where(obs.notnull())

    def compute_ci_and_significance(
           self, metric_name, metric_fn, members, obs, ens_dim='ensemble', 
           alpha=0.05, n_boot=1000
        ):
        """
        Compute 2D CI and significance mask for a given metric map (e.g., bias, RMSE).
        Returns (lower_CI, upper_CI, significance_mask)
        """
        try:
            ci_lower, ci_upper, signif_mask = self.bootstrap_ci_map(
                members, obs, metric_fn, ens_dim, n_boot, alpha
            )
            signif_mask.name = f'{metric_name}_significance_mask'
            ci_lower.name = f'{metric_name}_ci_lower'
            ci_upper.name = f'{metric_name}_ci_upper'

            signif_mask.attrs.update({'units': '1', 'long_name': f'Significance mask for {metric_name.replace("_", " ")}'})
            ci_lower.attrs.update({'units': self.var_unit, 'long_name': f'Lower bound of 95% CI for {metric_name.replace("_", " ")}'})
            ci_upper.attrs.update({'units': self.var_unit, 'long_name': f'Upper bound of 95% CI for {metric_name.replace("_", " ")}'})

            return ci_lower, ci_upper, signif_mask
        except Exception as e:
            print(f"[WARN] Failed CI + significance for {metric_name}: {e}")
            return None, None, None

    def bootstrap_confidence_interval(self, metric_fn, members, obs, ens_dim='ensemble', n_boot=1000, alpha=0.05):
        """Compute bootstrap confidence interval for an ensemble-based metric."""
        rng = np.random.default_rng(seed=42)
        n_ens = members.sizes[ens_dim]
        boot_vals = []

        for _ in range(n_boot):
            sample_idx = rng.integers(0, n_ens, size=n_ens)
            sample = members.isel({ens_dim: sample_idx})
            try:
                val = metric_fn(sample.mean(dim=ens_dim), obs, ens_dim = ens_dim)
                if np.isfinite(val):
                    boot_vals.append(val)
            except Exception:
                continue

        if not boot_vals:
            return np.nan, np.nan, np.nan

        boot_vals = np.array(boot_vals)
        lower = np.percentile(boot_vals, 100 * alpha / 2)
        upper = np.percentile(boot_vals, 100 * (1 - alpha / 2))
        return lower, upper, np.mean(boot_vals)

    def compute_rank_histogram(self, members: xr.DataArray, obs: xr.DataArray, normalize=False):
        ens_sorted = np.sort(members.values, axis=0)
        obs_val = obs.values
        n_ens = ens_sorted.shape[0]

        ranks = np.sum(ens_sorted < obs_val, axis=0).flatten()
        valid_ranks = ranks[(~np.isnan(ranks)) & (ranks >= 0) & (ranks <= n_ens)]
        hist, _ = np.histogram(valid_ranks, bins=np.arange(n_ens + 2) - 0.5, density=normalize)
        return xr.DataArray(hist, dims=['rank'], coords={'rank': np.arange(n_ens + 1)})

    def compute_ensemble_coverage(self, ens_dim: str, members: xr.DataArray, obs: xr.DataArray):
        ens_min = members.min(dim=ens_dim)
        ens_max = members.max(dim=ens_dim)
        within = ((obs >= ens_min) & (obs <= ens_max)).astype(float)

        return xr.DataArray(
            within.mean().compute().item(),
            attrs={
                'units': '1',
                'long_name': 'Fraction of domain where obs within ensemble envelope'
            }
        )

    def compute_crps(self, members: xr.DataArray, obs: xr.DataArray) -> xr.DataArray:
        try:
            ens_dim = self._get_ensemble_dim(members)
            # Ensure dims are ordered correctly and data types are preserved
            members = members.transpose(ens_dim, "lat", "lon")
            obs = obs.transpose("lat", "lon")
            # Compute CRPS using xskillscore (returns xarray.DataArray)
            crps = xs.crps_ensemble(obs, members, member_dim=ens_dim, dim=[])
            return crps.where(obs.notnull())
        except Exception as e:
            print(f"[WARN] CRPS computation failed: {e}")
            return xr.full_like(obs, np.nan)

    def _derive_hor_metrics_data(self, dso1, dsm1, ens_dim):
        # Chunk data for performance (if using dask-backed xarray)
        dsm1 = dsm1.chunk({dim: -1 for dim in dsm1.dims if dim in ['time', ens_dim, 'lat', 'lon']})
        dso1 = dso1.chunk({dim: -1 for dim in dso1.dims if dim in ['time', 'lat', 'lon']})
        
        mec_dict = {}
        bias, stddev, rmse, spread, dsm_mean = None, None, None, None, None
        
        ens_mean = dsm1.mean(dim=ens_dim, skipna=True)
        ens_std = dsm1.std(dim=ens_dim, skipna=True)
        
        dsm_mean = ens_mean.mean(dim='time', skipna=True) if 'time' in ens_mean.dims else ens_mean
        weights = self._generate_coslat_weight(dsm_mean)
        
        # Model global mean and RMS
        mec_dict['mod_mean'] = (dsm_mean * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True)
        mec_dict['mod_rms'] = np.sqrt((((dsm_mean - mec_dict['mod_mean']) ** 2) * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True))
        
        if isinstance(dso1, xr.DataArray):
            dso_mean = dso1.mean(dim='time', skipna=True) if 'time' in dso1.dims else dso1
            
            mec_dict['obs_mean'] = (dso_mean * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True)
            mec_dict['obs_rms'] = np.sqrt((((dso_mean - mec_dict['obs_mean']) ** 2) * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True))
            
            # Global RMSE and pattern correlation
            mec_dict['rmse_glb'] = xs.rmse(dso_mean, dsm_mean, dim=["lat", "lon"], weights=weights, skipna=True)
            mec_dict['pcor'] = xs.pearson_r(
                dso_mean - mec_dict['obs_mean'],
                dsm_mean - mec_dict['mod_mean'],
                dim=["lat", "lon"],
                weights=weights,
                skipna=True
            )
            
            # Bias (model mean - obs) averaged over time and space
            diff = ens_mean - dso1
            bias = diff.mean(dim='time', skipna=True) if 'time' in diff.dims else diff
            mec_dict['bias'] = (bias * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True)
            
            # Std dev of difference (observation error) and spread of ensemble
            stddev = diff.std(dim='time', skipna=True) if 'time' in diff.dims else xr.zeros_like(diff)
            spread = ens_std.mean(dim='time', skipna=True) if 'time' in ens_std.dims else ens_std
            mec_dict['spread'] = (spread * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True)
            
            # RMSE per ensemble member and spatial average
            dso1_expanded = dso1.expand_dims({ens_dim: dsm1[ens_dim]})
            rerr = xs.rmse(dsm1, dso1_expanded, dim=ens_dim, skipna=True)
            rmse = rerr.mean(dim='time', skipna=True) if 'time' in rerr.dims else rerr
            mec_dict['rmse'] = (rmse * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True)
        
        return mec_dict, bias, stddev, rmse, spread, dsm_mean

    def unify_fill_missing_da(
            self, da: xr.DataArray,
            sentinels=None
        ) -> xr.DataArray:
        
        da = da.copy()

        # Replace any sentinel values with NaN (keeps things CF-safe)
        for s in sentinels:
            try:
                da = da.where(da != s, np.nan)
            except Exception:
                pass

        # Also respect attrs if present
        for k in ("missing_value", "_FillValue"):
            v = da.attrs.get(k, None)
            if v is not None:
                try:
                    da = da.where(da != v, np.nan)
                except Exception:
                    pass

        # Remove conflicting metadata + encodings so they don't come back on save
        da.attrs.pop("missing_value", None)
        da.attrs.pop("_FillValue", None)
        da.encoding.pop("missing_value", None)
        da.encoding.pop("_FillValue", None)

        # Keep files small/consistent
        if np.issubdtype(da.dtype, np.floating):
            da = da.astype("float32")

        return da

    def derive_metrics_data(self):
        results = {}
        for model in self.model_dict:
            print(f"[INFO] Processing {model}")
            
            ens_dim = self._get_ensemble_dim(self.model_dict[model])
            mod_full = self.model_dict[model]
            obs_full = self.obs_data
            
            DEFAULT_SENTINELS = (-9999.0, -999.0, 1.0e20, 9.96921e36)
            mod_full = self.unify_fill_missing_da(mod_full,sentinels=DEFAULT_SENTINELS)
            obs_full = self.unify_fill_missing_da(obs_full,sentinels=DEFAULT_SENTINELS)

            if 'time' in mod_full.dims and mod_full.sizes['time'] == 1:
                mod_full = mod_full.squeeze('time')

            if 'time' in obs_full.dims and obs_full.sizes['time'] == 1:
                obs_full = obs_full.squeeze('time')

            metrics, bias_map, bias_std_map, rmse_map, spread_map, mean_map = self._derive_hor_metrics_data(
                obs_full, mod_full, ens_dim
            ) 
            
            mse = metrics['rmse'] ** 2
            bias_sq = metrics['bias'] ** 2
            spread_sq = metrics['spread'] ** 2
            sem = metrics['spread'] / np.sqrt(mod_full.sizes[ens_dim])
            metrics.update({
                'ssr': metrics['spread'] / metrics['rmse'],
                'bias_squared': bias_sq,
                'spread_squared': spread_sq,
                'mse': mse,
                'sem': sem
            })

            rank_hist = self.compute_rank_histogram(mod_full, obs_full)
            coverage = self.compute_ensemble_coverage(ens_dim, mod_full, obs_full)
            crps_map = self.compute_crps(mod_full, obs_full)
            weights = self._generate_coslat_weight(crps_map)
            crps_mean = (crps_map * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True)
            metrics.update({
                'rank_histogram': rank_hist,
                'coverage': coverage,
                'crps': crps_mean
            })

            # Bootstrap significance testing
            bias_fn = lambda mod, obs: float((mod - obs).mean(dim=["lat", "lon"], skipna=True).values)
            rmse_fn = lambda mod, obs: float(xs.rmse(obs, mod, dim=["lat", "lon"], skipna=True).values)
            spread_fn = lambda mod, obs: float(mod.std(dim=ens_dim, skipna=True).mean(dim=["lat", "lon"], skipna=True).values)
            pcor_fn = lambda mod, obs: float(xs.pearson_r(obs, mod, dim=["lat", "lon"], skipna=True).values)
            ssr_fn = lambda mod, obs: spread_fn(mod, obs) / rmse_fn(mod, obs)

            try:
                bias_lb, bias_ub, _ = self.bootstrap_confidence_interval(bias_fn, mod_full, self.obs_data, ens_dim)
                rmse_lb, rmse_ub, _ = self.bootstrap_confidence_interval(rmse_fn, mod_full, self.obs_data, ens_dim)
                spread_lb, spread_ub, _ = self.bootstrap_confidence_interval(spread_fn, mod_full, self.obs_data, ens_dim)
                pcor_lb, pcor_ub, _ = self.bootstrap_confidence_interval(pcor_fn, mod_full, self.obs_data, ens_dim)
                ssr_lb, ssr_ub, _ = self.bootstrap_confidence_interval(ssr_fn, mod_full, self.obs_data, ens_dim)

                metrics.update({
                    'bias_ci_lower': bias_lb,
                    'bias_ci_upper': bias_ub,
                    'rmse_ci_lower': rmse_lb,
                    'rmse_ci_upper': rmse_ub,
                    'spread_ci_lower': spread_lb,
                    'spread_ci_upper': spread_ub,
                    'pcor_ci_lower': pcor_lb,
                    'pcor_ci_upper': pcor_ub,
                    'ssr_ci_lower': ssr_lb,
                    'ssr_ci_upper': ssr_ub
                })
            except Exception as e:
                print(f"[WARN] Bootstrap CI failed: {e}")

            # Bias Map
            bias_ci_lo, bias_ci_hi, bias_ci_mask = self.compute_ci_and_significance(
                "bias_map", self.bias_map_fn, mod_full, obs_full, ens_dim)
            # RMSE Map
            rmse_ci_lo, rmse_ci_hi, rmse_ci_mask = self.compute_ci_and_significance(
                "rmse_map", self.rmse_map_fn, mod_full, obs_full, ens_dim)
            # Spread Map
            spread_ci_lo, spread_ci_hi, spread_ci_mask = self.compute_ci_and_significance(
                "spread_map", self.spread_map_fn, mod_full, obs_full, ens_dim)
            # CRPS Map
            crps_ci_lo, crps_ci_hi, crps_ci_mask = self.compute_ci_and_significance(
                "crps_map", self.crps_map_fn, mod_full, obs_full, ens_dim)

            results[model] = {
                'metrics': metrics,
                'mod_map': mod_full.mean(dim='time', skipna=True) if 'time' in mod_full.dims else mod_full,
                'obs_map': obs_full.mean(dim='time', skipna=True) if 'time' in obs_full.dims else obs_full,
                'mean_map': mean_map,
                'spread_map': spread_map,
                'bias_map': bias_map,
                'bias_stddev_map': bias_std_map,
                'rmse_map': rmse_map, 
                'crps_map': crps_map,
                'bias_map_ci_lower': bias_ci_lo,
                'bias_map_ci_upper': bias_ci_hi,
                'bias_map_significance_mask': bias_ci_mask,
                'rmse_map_ci_lower': rmse_ci_lo,
                'rmse_map_ci_upper': rmse_ci_hi, 
                'rmse_map_significance_mask': rmse_ci_mask,
                'spread_map_ci_lower': spread_ci_lo,
                'spread_map_ci_upper': spread_ci_hi,
                'spread_map_significance_mask': spread_ci_mask,
                'crps_map_ci_lower': crps_ci_lo, 
                'crps_map_ci_upper': crps_ci_hi,
                'crps_map_significance_mask': crps_ci_mask    
            }

            filepath_nc = os.path.join(
                self.output_path, 
                f"{model}_{self.var_name}_{self.reference}_ensemble_mean_bias.nc"
            )
            self.save_to_netcdf(filepath_nc, model, results[model])

            filepath_csv = os.path.join(
                self.output_path,
                f"{model}_{self.var_name}_{self.reference}_ensemble_bias_summary.csv"
            )
            self.save_summary_csv(filepath_csv, model, results[model])

        return results
    
    def save_to_netcdf(self, filepath, model, model_data):
        ds_out = {}
        metadata = {
            'mod_map': {'units': self.var_unit, 'long_name': f'{self.var_name} ensemble member (model)'},
            'obs_map': {'units': self.var_unit, 'long_name': f'{self.var_name} ensemble member ({self.reference})'},
            'pcor_map': {'units': '1', 'long_name': f'{self.var_name} pattern correlation (model vs {self.reference})'},
            'rmse_map': {'units': self.var_unit, 'long_name': f'{self.var_name} ensemble rmse (model vs {self.reference})'},
            'bias_map': {'units': self.var_unit, 'long_name': f'{self.var_name} ensemble mean bias (model - {self.reference})'},
            'spread_map': {'units': self.var_unit, 'long_name': f'{self.var_name} ensemble spread'},
            'mean_map': {'units': self.var_unit, 'long_name': f'{self.var_name} ensemble mean'},
            'bias_stddev_map': {'units': self.var_unit, 'long_name': f'{self.var_name} stddev of bias over time'},
            'bias_map_ci_lower': {'units': self.var_unit, 'long_name': 'Lower bound of CI for mean bias'},
            'bias_map_ci_upper': {'units': self.var_unit, 'long_name': 'Upper bound of CI for mean bias'},
            'bias_map_significance_mask': {'units': '1', 'long_name': 'Significance mask for mean bias'},
            'rmse_map_ci_lower': {'units': self.var_unit, 'long_name': 'Lower bound of CI for RMSE map'},
            'rmse_map_ci_upper': {'units': self.var_unit, 'long_name': 'Upper bound of CI for RMSE map'},
            'rmse_map_significance_mask': {'units': '1', 'long_name': 'Significance mask for RMSE'},
            'crps_map_ci_lower': {'units': self.var_unit, 'long_name': f'Lower bound of 95% CI for CRPS'},
            'crps_map_ci_upper': {'units': self.var_unit, 'long_name': f'Upper bound of 95% CI for CRPS'},
            'crps_map_significance_mask': {'units': '1', 'long_name': 'Significance mask for CRPS'},
            'spread_map_ci_lower': {'units': self.var_unit, 'long_name': f'Lower bound of 95% CI for CRPS'},
            'spread_map_ci_upper': {'units': self.var_unit, 'long_name': f'Upper bound of 95% CI for CRPS'},
            'spread_map_significance_mask': {'units': '1', 'long_name': 'Significance mask for ensemble spread'},
            'mean': {'units': self.var_unit, 'long_name': f'{self.var_name} area-weighted mean (model)'},
            'rmsm': {'units': self.var_unit, 'long_name': f'{self.var_name} global RMS (model)'},
            'rmso': {'units': self.var_unit, 'long_name': f'{self.var_name} global RMS (observation)'},
            'rmse': {'units': self.var_unit, 'long_name': f'{self.var_name} global mean ensemble RMSE (model vs {self.reference})'},
            'rmse_glb': {'units': self.var_unit, 'long_name': f'{self.var_name} ensemble mean global RMSE (model vs {self.reference})'},
            'spread': {'units': self.var_unit, 'long_name': f'{self.var_name} spatial mean ensemble spread'},
            'ssr': {'units': '1', 'long_name': 'Spread-to-RMSE ratio'},
            'bias': {'units': f'{self.var_unit}^2', 'long_name': 'Spatial mean ensemble mean bias'},
            'bias_squared': {'units': f'{self.var_unit}^2', 'long_name': 'Squared ensemble mean bias'},
            'spread_squared': {'units': f'{self.var_unit}^2', 'long_name': 'Squared ensemble spread'},
            'mse': {'units': f'{self.var_unit}^2', 'long_name': 'Mean square error (bias² + spread²)'},
            'sem': {'units': self.var_unit, 'long_name': 'Standard error of the ensemble mean'},
            'bias_ci_lower': {'units': self.var_unit, 'long_name': 'Lower bound of 95% CI for bias'},
            'bias_ci_upper': {'units': self.var_unit, 'long_name': 'Upper bound of 95% CI for bias'},
            'rmse_ci_lower': {'units': self.var_unit, 'long_name': 'Lower bound of 95% CI for RMSE'},
            'rmse_ci_upper': {'units': self.var_unit, 'long_name': 'Upper bound of 95% CI for RMSE'},
            'spread_ci_lower': {'units': self.var_unit, 'long_name': 'Lower bound of 95% CI for spread'},
            'spread_ci_upper': {'units': self.var_unit, 'long_name': 'Upper bound of 95% CI for spread'},
            'pcor_ci_lower': {'units': '1', 'long_name': 'Lower bound of 95% CI for pattern correlation'},
            'pcor_ci_upper': {'units': '1', 'long_name': 'Upper bound of 95% CI for pattern correlation'},
            'ssr_ci_lower': {'units': '1', 'long_name': 'Lower bound of 95% CI for spread-to-RMSE ratio'},
            'ssr_ci_upper': {'units': '1', 'long_name': 'Upper bound of 95% CI for spread-to-RMSE ratio'}, 
            'rank_histogram': {'units': 'count', 'long_name': 'Rank histogram of observation within ensemble'},
            'coverage': {'units': '1', 'long_name': 'Fraction of domain where obs within ensemble envelope'},
            'crps': {'units': self.var_unit, 'long_name': f'CRPS: Continuous Ranked Probability Score for {self.var_name}'},
            'crps_mean': {'units': self.var_unit, 'long_name': f'Mean CRPS over spatial domain for {self.var_name}'}
        }

        for key, val in model_data['metrics'].items():
            da = val if isinstance(val, xr.DataArray) else xr.DataArray(val)
            # Clean dimension conflicts
            if 'time' in da.coords and 'time' not in da.dims:
                da = da.reset_coords('time', drop=True)
            if da.ndim == 0:
                da = da.expand_dims({'scalar_dim': [0]})
            if key in metadata:
                da.attrs.update(metadata[key])
            ds_out[key] = da

        for key, val in model_data.items():
            if isinstance(val, xr.DataArray):
                if 'time' in val.coords and 'time' not in val.dims:
                    val = val.reset_coords('time', drop=True)
                val.attrs.update(metadata.get(key, {}))
                ds_out[key] = val

        ds = xr.Dataset(ds_out)
        ds.attrs.update({
            'model': model,
            'component': self.component,
            'reference_dataset': self.reference,
            'variable': self.var_name,
            'units': self.var_unit,
            'created_by': 'EnsembleMetricEvaluator',
            'creation_time': datetime.now().isoformat()
        })

        ds.to_netcdf(filepath)
        print(f"[INFO] Saved metrics for {model} to {filepath}")

    def safe_mean(self, val):
        if isinstance(val, xr.DataArray):
            try:
                mean_val = val.mean(skipna=True).values
                return float(mean_val) if not np.isnan(mean_val) else np.nan
            except Exception:
                return np.nan
        elif isinstance(val, (int, float, np.number)):
            return float(val)
        else:
            return np.nan

    def save_summary_csv(self, filepath, model, model_data):
        summary = []
        metrics = model_data['metrics']
        row = {
            'model': model,
            'RMSE': self.safe_mean(metrics.get('rmse')),
            'RMSE_GLB': self.safe_mean(metrics.get('rmse_glb')),
            'PCOR': self.safe_mean(metrics.get('pcor')),
            'Spread': self.safe_mean(metrics.get('spread')),
            'SSR': self.safe_mean(metrics.get('ssr')),
            'Bias^2': self.safe_mean(metrics.get('bias_squared')),
            'Spread^2': self.safe_mean(metrics.get('spread_squared')),
            'MSE': self.safe_mean(metrics.get('mse')),
            'SEM': self.safe_mean(metrics.get('sem')),
            'Coverage': self.safe_mean(metrics.get('coverage')),
            'CRPS': self.safe_mean(metrics.get('crps_mean')),
            'Bias_CI_Lower': metrics.get('bias_ci_lower'),
            'Bias_CI_Upper': metrics.get('bias_ci_upper'),
            'RMSE_CI_Lower': metrics.get('rmse_ci_lower'),
            'RMSE_CI_Upper': metrics.get('rmse_ci_upper'),
            'Spread_CI_Lower': metrics.get('spread_ci_lower'),
            'Spread_CI_Upper': metrics.get('spread_ci_upper'),
            'PCOR_CI_Lower': metrics.get('pcor_ci_lower'),
            'PCOR_CI_Upper': metrics.get('pcor_ci_upper'),
            'SSR_CI_Lower': metrics.get('ssr_ci_lower'),
            'SSR_CI_Upper': metrics.get('ssr_ci_upper')
        }

        if all(pd.isna(v) for k, v in row.items() if k != 'model'):
            print(f"[WARN] No valid metrics found for {model}, CSV will be empty.")
            return

        summary.append(row)
        df = pd.DataFrame(summary)
        if os.path.exists(filepath):
            os.remove(filepath)
        df.to_csv(filepath, index=False)
        print(f"[INFO] Saved summary CSV to {filepath}")

In [3]:
@staticmethod
def extract_exp_info(base_name, data_path) -> dict:
    exps = {
        'CTRLEN10':       {'name': 'ctrl_en10', 'nens': 10, 'period': '201112-201112'},
        'DARTEN10':       {'name': 'dart_en10', 'nens': 10, 'period': '201112-201112'},
        'DARTEN20':       {'name': 'dart_en20', 'nens': 20, 'period': '201112-201112'},
        'DARTEN40':       {'name': 'dart_en40', 'nens': 40, 'period': '201112-201112'},
        'CAPTEN10_15day': {'name': 'capt_en10', 'nens': 10, 'period': '201201-201202'},
        'DARTEN20_15day': {'name': 'dart_en20', 'nens': 20, 'period': '201201-201202'},
        'DARTEN40_15day': {'name': 'dart_en40', 'nens': 40, 'period': '201201-201202'},
    }
    return {
        exp_name: {
            'run': f"{exp_name}_{base_name}" if base_name else exp_name,
            'key': exp_data['name'],
            'nens': exp_data['nens'],
            'atm': 'archive/post/atm/180x360_aave',
            'lnd': 'archive/post/lnd/180x360_aave',
            'period': exp_data['period']
        }
        for exp_name, exp_data in sorted(exps.items())
    }

In [ ]:
if __name__ == "__main__":
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    fig_path = "./figures"
    
    # Ensure output path exists
    os.makedirs(out_path, exist_ok=True)

    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    regnam = "global"
    plot_type = "model_vs_obs"
    period = "201112-201112" 
    frequency = "monthly"
    derive_monthly = True 
    
    # === Variable dictionary ===
    var_dict = {
        'FLUT':   {'alias' : 'FLUT',   'unit': 'W m$^{-2}$',    'level': np.linspace(-10, 10, 51), 'comp': 'atm', 'ref': "CERES-OAFlux"},
        'PRECT':  {'alias' : 'PRECT',  'unit': 'mm day$^{-1}$', 'level': np.linspace(-2, 2, 51),   'comp': 'atm', 'ref': "GPCP"},
        'TMQ':    {'alias' : 'TMQ',    'unit': 'kg m$^{-2}$',   'level': np.linspace(-10, 10, 51), 'comp': 'atm', 'ref': "ERA5"},
        'TS':     {'alias' : 'TS',     'unit': '$^o$C',         'level': np.linspace(-8, 8, 51),   'comp': 'atm', 'ref': "ERA5"},
        'TREFHT': {'alias' : 'TREFHT', 'unit': '$^o$C',         'level': np.linspace(-8, 8, 51),   'comp': 'atm', 'ref': "ERA5"},
        'PSL':    {'alias' : 'PSL',    'unit': 'hPa',           'level': np.linspace(-10, 10, 51), 'comp': 'atm', 'ref': "ERA5"},
    }
    
    exp_dict = extract_exp_info(exp_base, top_path)
    
    model_dict = {
        'CTRLEN10': "CTRL (ENS=10)",
        #'DARTEN10': "EAM-DART (ENS=10)",
        'DARTEN20': "EAM-DART (ENS=20)",
        'DARTEN40': "EAM-DART (ENS=40)",
    }

    model_list = list(model_dict.keys())

    # Optional: save run configuration for reproducibility
    config_file = os.path.join(out_path, "diagnostic_run_config.txt")
    with open(config_file, "w") as f:
        f.write(f"Run Time: {datetime.now().isoformat()}\n")
        f.write(f"Variables: {list(var_dict.keys())}\n")
        f.write(f"Models: {model_list}\n")
        f.write(f"Region: {regnam}, Period: {period}, Frequency: {frequency}\n")
        f.write(f"Experiment Base: {exp_base}\n")

    # === Loop over variables (refactored: process one experiment at a time) ===
    for var, meta in var_dict.items():
        try:
            print(f"[{datetime.now().isoformat()}] Variable {var} | ref={meta.get('ref','ERA5')}")
            component    = meta.get('comp', 'atm')
            ref_var      = meta.get('alias', var)
            ref_dataset  = meta.get('ref', 'ERA5')
            vunit        = meta['unit']

            # --- Load reference data once per variable
            try:
                obs_reader = ObsDataReader(
                    obsname=ref_dataset,
                    period=period,
                    frequency=frequency
                )
                obs = obs_reader.read(var=ref_var)
            except FileNotFoundError:
                print(f"[WARN] Skipping {var} — obs not found for {ref_var} ({ref_dataset})")
                continue

            # --- Prepare a model reader (reused per-var)
            model_reader = ModelDataReader(
                base_path=top_path,
                exp_dict=exp_dict,
                regnam=regnam,
                frequency=frequency,
                period=period,
                component=component,
                derive_monthly=derive_monthly
            )

            # --- Iterate experiments one-by-one
            for model_key in model_list:
                model_label = model_dict.get(model_key, model_key)
                t0 = datetime.now()
                print(f"[{t0.isoformat()}]   -> Reading {model_key} ({model_label}) for {var}")

                # Per-experiment output folder
                out_path_exp = os.path.join(out_path, model_key)
                os.makedirs(out_path_exp, exist_ok=True)

                # Try to read ONLY this experiment’s data
                try:
                    # If you have a dedicated single-exp reader, prefer it:
                    #   arr = model_reader.read_variable_for_experiment(var, model_key)
                    # Otherwise, reuse the plural API with a single item:
                    data_dict = model_reader.read_variable_across_experiments(var, [model_key])
                    if not data_dict or model_key not in data_dict:
                        print(f"[WARN] No data returned for {var} in {model_key}; skipping.")
                        continue
                    arr = data_dict[model_key]
                except FileNotFoundError:
                    print(f"[WARN] Missing files for {var} in {model_key}; skipping.")
                    continue
                except Exception as e:
                    print(f"[WARN] Failed reading {var} in {model_key}: {e}; skipping.")
                    continue

                # Evaluate metrics for this single experiment
                try:
                    evaluator = EnsembleMetricEvaluator(
                        obs=obs,
                        model_dict={model_key: arr},  # pass single entry to keep API unchanged
                        component=component,
                        ref_dataset=ref_dataset,
                        var=var,
                        vunit=vunit,
                        output_path=out_path_exp
                    )
                    # If the evaluator requires an explicit call to run/save, do it here, e.g.:
                    # evaluator.run_all() or evaluator.save_figures(fig_path=fig_path) ...
                except Exception as e:
                    print(f"[ERROR] Metric eval failed for {var} in {model_key}: {e}")
                    continue

                # Optional: per-exp config breadcrumb
                try:
                    cfg = os.path.join(out_path_exp, f"run_config_{var}.txt")
                    with open(cfg, "w") as f:
                        f.write(f"Run Time: {datetime.now().isoformat()}\n")
                        f.write(f"Variable: {var}\n")
                        f.write(f"Alias: {ref_var}\n")
                        f.write(f"Unit: {vunit}\n")
                        f.write(f"Region: {regnam}, Period: {period}, Frequency: {frequency}\n")
                        f.write(f"Experiment: {model_key} ({model_label})\n")
                        f.write(f"Component: {component}\n")
                        f.write(f"Reference: {ref_dataset}\n")
                except Exception as e:
                    print(f"[WARN] Could not write per-exp config for {var}/{model_key}: {e}")

                # Free memory between experiments (useful if xarray/dask)
                try:
                    import gc
                    del arr
                    gc.collect()
                except Exception:
                    pass

                print(f"[{datetime.now().isoformat()}]   -> Done {model_key} ({model_label}) for {var}")

            # Free obs after finishing this variable
            try:
                import gc
                del obs
                gc.collect()
            except Exception:
                pass
            
        except Exception as e:
            print(f"[ERROR] Failed to process {var}: {e}")


[2025-11-05T00:53:54.102084] Variable FLUT | ref=CERES-OAFlux
Parsed time range: 2011-12-01 00:00:00 to 2011-12-01 23:59:59
2011


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CLDTOT' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CRE' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CRE_SRF' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'EVAP' has multiple 

Extracting time range: 2011-12-01 00:00:00 to 2011-12-01 00:00:00
[2025-11-05T00:53:58.932778]   -> Reading CTRLEN10 (CTRL (ENS=10)) for FLUT
Parsed time range: 2011-12-01 00:00:00 to 2011-12-01 23:59:59
[INFO] Processing CTRLEN10
[INFO] Saved metrics for CTRLEN10 to /compyfs/zhan391/v3_dart_cda_scratch/diag_dart/CTRLEN10/CTRLEN10_FLUT_CERES-OAFlux_ensemble_mean_bias.nc
[INFO] Saved summary CSV to /compyfs/zhan391/v3_dart_cda_scratch/diag_dart/CTRLEN10/CTRLEN10_FLUT_CERES-OAFlux_ensemble_bias_summary.csv
[2025-11-05T01:05:06.241168]   -> Done CTRLEN10 (CTRL (ENS=10)) for FLUT
[2025-11-05T01:05:06.241466]   -> Reading DARTEN20 (EAM-DART (ENS=20)) for FLUT
Parsed time range: 2011-12-01 00:00:00 to 2011-12-01 23:59:59
[INFO] Processing DARTEN20
[INFO] Saved metrics for DARTEN20 to /compyfs/zhan391/v3_dart_cda_scratch/diag_dart/DARTEN20/DARTEN20_FLUT_CERES-OAFlux_ensemble_mean_bias.nc
[INFO] Saved summary CSV to /compyfs/zhan391/v3_dart_cda_scratch/diag_dart/DARTEN20/DARTEN20_FLUT_CERES-OA

In [ ]:
if __name__ == "__main__":
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    fig_path = "./figures"
    
    # Ensure output path exists
    os.makedirs(out_path, exist_ok=True)

    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    regnam = "global"
    plot_type = "model_vs_obs"
    period = "201112-201112" 
    frequency = "monthly"
    derive_monthly = True 
    
    # === Variable dictionary ===
    var_dict = {
        'FLUT':   {'alias' : 'FLUT',   'unit': 'W m$^{-2}$',    'level': np.linspace(-10, 10, 51), 'comp': 'atm', 'ref': "CERES-OAFlux"},
        'PRECT':  {'alias' : 'PRECT',  'unit': 'mm day$^{-1}$', 'level': np.linspace(-2, 2, 51),   'comp': 'atm', 'ref': "GPCP"},
        'TMQ':    {'alias' : 'TMQ',    'unit': 'kg m$^{-2}$',   'level': np.linspace(-10, 10, 51), 'comp': 'atm', 'ref': "ERA5"},
        'TS':     {'alias' : 'TS',     'unit': '$^o$C',         'level': np.linspace(-8, 8, 51),   'comp': 'atm', 'ref': "ERA5"},
        'TREFHT': {'alias' : 'TREFHT', 'unit': '$^o$C',         'level': np.linspace(-8, 8, 51),   'comp': 'atm', 'ref': "ERA5"},
        'PSL':    {'alias' : 'PSL',    'unit': 'hPa',           'level': np.linspace(-10, 10, 51), 'comp': 'atm', 'ref': "ERA5"},
    }
    
    exp_dict = extract_exp_info(exp_base, top_path)
    
    model_dict = {
        'CTRLEN10': "CTRL (ENS=10)",
        #'DARTEN10': "EAM-DART (ENS=10)",
        'DARTEN20': "EAM-DART (ENS=20)",
        'DARTEN40': "EAM-DART (ENS=40)",
    }

    model_list = list(model_dict.keys())

    # Optional: save run configuration for reproducibility
    config_file = os.path.join(out_path, "diagnostic_run_config.txt")
    with open(config_file, "w") as f:
        f.write(f"Run Time: {datetime.now().isoformat()}\n")
        f.write(f"Variables: {list(var_dict.keys())}\n")
        f.write(f"Models: {model_list}\n")
        f.write(f"Region: {regnam}, Period: {period}, Frequency: {frequency}\n")
        f.write(f"Experiment Base: {exp_base}\n")

    # === Loop over variables ===
    
    for var, meta in var_dict.items():
        try:
            print(f"[{datetime.now().isoformat()}] Processing {var} using {meta['ref']}")

            component = meta.get('comp', 'atm')
            ref_var = meta.get('alias', var)
            ref_dataset = meta.get('ref', 'ERA5')

            # Load reference data
            try:
                obs_reader =  ObsDataReader(
                    obsname=ref_dataset,
                    period=period,
                    frequency=frequency
                )
                obs = obs_reader.read(var=ref_var)
            except FileNotFoundError:
                print(f"[WARN] Skipping {var} — observation not found for {ref_var} ({ref_dataset})")
                continue
                
            # Load model ensemble data
            model_reader = ModelDataReader(
                base_path=top_path,
                exp_dict=exp_dict,
                regnam=regnam,
                frequency=frequency,
                period=period,
                component=component,
                derive_monthly=derive_monthly
            )
            data_dict = model_reader.read_variable_across_experiments(
                var, 
                model_list
            )
            #print(data_dict)
            #print(obs.min(),obs.max())
            #print(xxxx)
            
            # Evaluate metrics and save
            evaluator = EnsembleMetricEvaluator(
                obs=obs,
                model_dict=data_dict,
                component=component,
                ref_dataset=ref_dataset,
                var=var,
                vunit=meta['unit'],
                output_path=out_path
            )

        except Exception as e:
            print(f"[ERROR] Failed to process {var}: {e}")
